# 3주차 · 추진 시스템

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gabraxas/LVs-and-Policy/blob/main/lecture/week03/week03.ipynb)

> 이 노트북은 GitHub에 저장되고 Colab에서 실행됩니다. 위 배지를 눌러 Colab에서 열거나, 아래 첫 코드 셀부터 순서대로 실행하세요.

In [ ]:
# ▶ 실행 전 준비 — 이 셀을 먼저 실행하세요 (약 10~20초 소요)
!pip install -q ipywidgets
!apt-get -qq install -y fonts-nanum > /dev/null 2>&1

import urllib.request
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/_shared/course_interactive.py",
    "course_interactive.py")

from course_interactive import *
setup_korean_font()
print("준비 완료 — 아래 셀들을 순서대로 실행하며 강의를 진행하세요.")


# 3주차 · 추진 시스템

### Propulsion Systems — 로켓 엔진은 어떻게 작동하고, 왜 그렇게 설계되는가

> **우주수송정책과 발사체 기술** — Week 03
> 참고: Sutton & Biblarz, *Rocket Propulsion Elements* · Stanford AA284a *Advanced Rocket Propulsion*
> 서술 구조 참고: [ERAU, *Introduction to Aerospace Flight Vehicles* — "Rockets & Launch Vehicle Performance"](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/rocket-performance/)

---

## 서론

- 2주차: "얼마나 빨라져야 하는가(Δv)"를 다룸
- 3주차: **"그 속도를 만드는 기계"**를 다룸
- 지난주 로켓방정식에서 $I_{sp}$는 '주어진 값'이었음 → 오늘은 그 값이 어떻게 만들어지는지를 다룸

$$
\Delta v = I_{sp} \cdot g_0 \cdot \ln(m_0 / m_f)
$$

| 질문 | 답이 놓인 곳 | 다루는 곳 |
|---|---|---|
| $I_{sp}$를 결정하는 것 | 연소가스의 온도·분자량, 노즐 팽창비 | Part I |
| 온도·분자량을 결정하는 것 | 추진제 조합 (케로신/수소/메탄/고체) | Part II |
| 연소실 압력을 결정하는 것 | 추진제를 밀어넣는 방식 = 엔진 사이클 | Part III |
| 실제 운용 성능을 결정하는 것 | 재점화·스로틀링·냉각·수명 | Part IV |

> **직관**: $I_{sp}$는 "연료 1 kg으로 얼마나 오래 1 kgf의 힘을 낼 수 있는가"(초 단위) — 자동차 연비와 같은 개념. 값이 클수록 같은 추진제로 더 큰 Δv를 얻는다.

**오늘의 한 문장**: 엔진 설계는 "성능을 얼마나 끌어올릴 것인가"가 아니라 "어느 정도의 성능을 어느 정도의 위험과 비용으로 살 것인가"의 문제다.

## 학습 목표

- [ ] **비추력이 어디서 나오는지 설명한다** — 추력식의 두 항(운동량·압력) 구분, 노즐과 팽창비·고도의 관계, $I_{sp} \propto \sqrt{T_c/M}$의 의미
- [ ] **추진제 조합을 근거로 고를 수 있다** — 고체·액체·하이퍼골릭의 용도 구분, 케로신·수소·메탄의 장단점, 밀도비추력으로 단(stage)별 최적 선택
- [ ] **엔진 사이클의 트레이드오프를 읽는다** — 가스발생기·다단연소·전량연소·팽창기, 성능 이득 vs 개발 난이도·비용·일정, 전기펌프 등 대안 경로
- [ ] **정책 판단으로 연결한다** — 재사용이 요구하는 엔진 특성, 엔진이 왜 발사체 개발의 임계경로인가, KSLV-III의 메탄·사이클 선택 논거 해석

---

## PART I · 추진의 물리

### 1.1 추력은 어떻게 생기는가 — 두 개의 항

- 로켓은 무언가를 밀고 나아가지 않음 → 질량을 뒤로 던진 반작용으로 전진
- **직관 — 스케이트보드**: 보드 위에서 공을 뒤로 던지면 앞으로 밀림. 무거운 공을 더 빠르게, 더 자주 던질수록 세게 밀림. 로켓은 초당 수백 kg의 가스를 던짐

$$
F = \dot m \cdot V_e + (P_e - P_a)\cdot A_e
$$

① 운동량 항 — 던지는 양 × 던지는 속도 / ② 압력 항 — 노즐 출구와 바깥 공기의 압력 차

<img src="https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/w03_01.png" width="400">

- **① 운동량 항**
  - $\dot m$: 초당 뿜는 추진제 질량 / $V_e$: 배기가스 속도 (2~4.5 km/s)
  - 추력의 90% 이상을 담당
  - $V_e$를 키우는 것이 곧 $I_{sp}$ 향상
- **② 압력 항**
  - 지상: $P_a$ = 101 kPa → 손해 / 진공: $P_a$ = 0 → 이득
  - 같은 엔진도 고도가 오르면 추력 증가 (Merlin 1D: 지상 85 tf → 진공 93 tf)
- **왜 중요한가**
  - 1단은 지상 추력이, 상단은 진공 $I_{sp}$가 승부처
  - 따라서 1단과 상단은 애초에 다른 엔진이 됨
  - 이 구분이 뒤에서 노즐·추진제·사이클 선택을 모두 지배

### 1.2 노즐 — 왜 잘록해졌다가 다시 넓어지는가

- 연소실 = 뜨겁고 압력이 높지만 느림 → 노즐의 임무 = 그 열에너지를 방향이 정해진 속도로 전환
- 아음속: 통로를 좁혀야 가속 / 초음속: 통로를 넓혀야 가속 → 모래시계 모양

- **노즐이 만드는 차이**
  - 노즐 없이 구멍만 뚫으면 $V_e \approx$ 1 km/s 수준
  - 제대로 설계된 노즐: $V_e \approx$ 3~4.5 km/s
  - 노즐만으로 $I_{sp}$가 3배 이상 벌어짐 — **노즐은 '부속품'이 아니라 성능의 절반**

| 개념 | 의미 |
|---|---|
| 팽창비 $\varepsilon = A_e/A_t$ | 출구 면적 ÷ 목 면적. 가스를 얼마나 더 팽창시킬지를 정하는 단 하나의 형상 변수 |
| 1단 노즐 | $\varepsilon \approx$ 10~20. 지상 대기압에서도 흐름이 떨어지지 않도록 짧고 통통하게 |
| 상단·진공 노즐 | $\varepsilon \approx$ 80~300. 바깥이 진공이므로 끝까지 팽창 — 나팔처럼 거대해짐 |

### 1.3 과팽창과 부족팽창 — 완벽한 노즐은 없다

- 노즐은 오직 한 고도에서만 최적 → 발사체는 지상~진공을 관통하므로 어느 쪽으로든 손해 감수

In [ ]:
# 02-nozzle-expansion-regimes.png 대신 인터랙티브 위젯으로 대체됨
thrust_nozzle_explorer()

| 상태 | 조건 | 설명 | 예 |
|---|---|---|---|
| 과팽창 (Over-expanded) | $P_e < P_a$ | 노즐이 너무 큼. 배기가 바깥 공기에 눌려 수축, 심하면 벽에서 박리(흐름 이탈)되어 옆방향 하중 발생 | 지상 이륙 직후의 진공용 노즐 |
| 최적 팽창 (Optimum) | $P_e = P_a$ | 출구압=외기압, 압력항 0, 그 고도에서 추력 최대 | 설계 고도 — 단 한 지점 |
| 부족팽창 (Under-expanded) | $P_e > P_a$ | 노즐이 너무 작음. 가스가 빠져나온 뒤에도 계속 퍼지며 에너지를 추력으로 회수하지 못함 | 고고도의 1단 노즐 |

- **설계자의 타협**
  - 1단 노즐은 지상에서 약간 과팽창되도록 설계 (대부분의 비행 시간이 고고도이기 때문)
  - 상단 엔진은 아예 진공에서만 점화하도록 임무 분리
  - → 1단·2단 엔진이 '같은 엔진의 노즐만 바꾼' 형태가 흔함 (Merlin 1D / Merlin Vacuum)
- **고도보상 노즐이라는 오랜 꿈**
  - 에어로스파이크: 대기압이 스스로 노즐 벽 역할 → 전 고도 최적
  - 이론적 이득은 크지만 냉각 면적·질량·시험 난이도가 커서 실용화 사례 없음
  - **"성능은 좋지만 개발할 수 없는 기술"의 전형** — 오늘 강의의 반복 주제

### 1.4 Isp를 올리는 방법은 단 두 가지

- 배기속도 = 결국 "뜨거운가"와 "가벼운가"의 싸움

$$
V_e \propto \sqrt{T_c / M}
$$

$T_c$: 연소실 온도(뜨거울수록 좋음) / $M$: 배기가스 평균 분자량(가벼울수록 좋음)

- **T — 온도를 올린다 (한계 있음)**
  - 연소온도 3,000 K 이상에서는 가스가 해리(dissociation)되어 에너지가 되돌아옴
  - 무엇보다 벽면이 녹음 — 냉각 기술이 한계를 정함
  - 즉 온도는 재료·냉각의 문제이지 화학의 문제가 아님
- **M — 분자량을 낮춘다 (훨씬 효과적)**
  - 수소 연소 배기(H₂O + 잉여 H₂)의 평균 분자량은 매우 낮음
  - LOX/LH2는 연소온도가 더 낮은데도 $I_{sp}$가 가장 높은 이유
  - 실무에서는 일부러 연료를 과잉 공급해 M을 낮춤

In [ ]:
# 03-propellant-isp-comparison.png 대신 인터랙티브 위젯으로 대체됨
propellant_isp_explorer()

| 추진제 조합 | 연소온도 | 배기 분자량 | 진공 Isp(대표값) | 해석 |
|---|---|---|---|---|
| LOX / LH2 | ≈ 3,000 K | 낮음 (≈ 10) | 450 s | 온도는 중간, 분자량이 압도적으로 유리 |
| LOX / CH4 | ≈ 3,500 K | 중간 (≈ 20) | 360 s | 온도·분자량 모두 무난한 균형점 |
| LOX / RP-1 | ≈ 3,600 K | 높음 (≈ 22) | 340 s | 가장 뜨겁지만 무거운 탄소 때문에 손해 |
| 고체 (APCP) | ≈ 3,000 K | 매우 높음 (≈ 25) | 265 s | 금속 연료·염화물 때문에 배기가 무거움 |

> ※ 값은 공개자료 기준의 대표적 근사치이며 혼합비·팽창비에 따라 달라진다.

### 1.5 엔진 성적표를 두 과목으로 나누기 — c*와 C_F

- $I_{sp}$ 하나만 보면 어디가 잘못됐는지 알 수 없음 → 실무는 성능을 연소기 몫과 노즐 몫으로 분리

$$
I_{sp} \cdot g_0 = c^* \times C_F
$$

$c^*$(특성속도) = 연소가 얼마나 잘 됐나 / $C_F$(추력계수) = 노즐이 얼마나 잘 뽑아냈나

![diagram](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/04-cstar-cf-breakdown.png)

- **c\* — 연소기의 성적**
  - $c^* = P_c \cdot A_t / \dot m$
  - 추진제 조합·혼합비·분사기(injector) 혼합 품질로 결정, 노즐 형상과 무관
  - 실제 c* ÷ 이론 c* = 연소효율 (통상 95~99% 목표, 낮으면 분사기 재설계)
- **C_F — 노즐의 성적**
  - $F = C_F \cdot P_c \cdot A_t$
  - 팽창비와 바깥 압력(고도)만으로 거의 결정, 추진제 종류와 거의 무관
  - 통상 1.5~1.9 — 노즐은 목에서 나오는 힘을 1.5~1.9배로 증폭하는 장치
- **왜 이렇게 나누는가**
  - 성능 미달 시 원인 특정 가능: c* 낮음 → 분사기·연소 문제 / C_F 낮음 → 노즐 문제
  - 개발 현장에서 연소기 시험과 노즐 시험을 분리 수행 → 시험설비 투자 항목의 분리로 이어짐

---

## PART II · 추진제 선택

### 2.1 세 가지 계보 — 고체 · 액체 · 하이브리드

- 성능만 보면 액체가 앞섬 → 그런데도 고체가 사라지지 않는 이유: 성능이 유일한 기준이 아니기 때문

| 구분 | 원리 | 장점 | 단점 | 주 용도 |
|---|---|---|---|---|
| **고체(Solid)** | 연료+산화제를 고무 형태로 굳혀 케이스에 채움 | 즉시 점화, 수년간 보관, 구조 단순→신뢰도 높음, 밀도 높음 | 한번 켜면 끌 수 없음, $I_{sp}$ 낮음(≈265s) | 미사일, 부스터, 킥모터 |
| **액체(Liquid)** | 연료+산화제를 별도 탱크에 담고 펌프로 연소실에 보냄 | $I_{sp}$ 높음(340~460s), 정지·재점화·추력조절 가능 | 탱크·펌프·배관으로 복잡, 극저온이면 상시대기 불가 | 거의 모든 궤도발사체 |
| **하이브리드(Hybrid)** | 고체 연료 + 액체(기체) 산화제의 절충 | 산화제 밸브로 추력조절·정지 가능, 폭발위험 낮아 안전 | 연소면 후퇴로 성능 변화, 대형화 실적 부족 | 우주관광·표적기(SpaceShipTwo 등) |

> **정책적 함의**: 고체 기술은 그대로 탄도미사일 기술이다. 7주차 MTCR·수출통제의 실질적 대상은 대부분 고체 추진과 관련 소재·설비다.

### 2.2 고체 추진 — 형상이 곧 추력 곡선이다

- 액체 엔진은 밸브로 추력 조절 / 고체는 그럴 수 없어 연료 덩어리를 어떤 모양으로 깎을지로 미리 결정

![diagram](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/05-solid-grain-thrust-profiles.png)

| 그레인 단면 | 추력 프로파일 |
|---|---|
| 원통형 | 연소면이 넓어지며 추력이 점점 증가 |
| 별 모양 | 연소면이 거의 일정, 추력이 평탄 |
| 끝면 연소 | 연소면이 좁아지며 추력이 감소 |

- **고체가 선택되는 이유**
  - 대기성(readiness): 충전된 상태로 수년간 보관 → 군용 미사일의 필수 조건
  - 밀도: $\rho \approx$ 1.8 g/cm³로 액체보다 압도적 — 좁은 발사관·잠수함에 유리
  - 단순성: 펌프·배관·극저온 설비가 없어 부품 수가 한 자릿수 단위
  - 부스터 활용: 이륙 순간의 큰 추력만 담당 (Ariane 6, H3, SLS, Vega)

| 한계 영역 | 내용 |
|---|---|
| 제어 | 점화 후 정지·재점화 불가. 종료는 파괴형 추력종단(thrust termination)으로만 가능 |
| 성능 | $I_{sp} \approx$ 265 s로 액체 대비 25~40% 낮음 → 같은 임무에 더 큰 추진제 질량 필요 |
| 안전·취급 | 추진제 자체가 폭발물. 제조·저장·운송이 화약류 규제 대상 |
| 비용 구조 | 생산 단가는 낮으나 대형 그레인 주조 설비가 진입장벽 — 소수 업체 과점 |

### 2.3 액체 추진제 3파전 — 케로신 · 수소 · 메탄

- 산화제는 사실상 액체산소(LOX)로 고정 → 실질적 선택은 연료

| 비교 항목 | LOX/RP-1(케로신) | LOX/LH2(수소) | LOX/CH4(메탄) |
|---|---|---|---|
| 진공 Isp(대표값) | ≈ 340 s | ≈ 450 s | ≈ 360 s |
| 혼합 벌크 밀도 | ≈ 1.02 g/cm³ | ≈ 0.36 g/cm³ | ≈ 0.83 g/cm³ |
| 연료 보관 온도 | 상온 | −253°C(극저온 심화) | −162°C |
| 탱크 부피 | 가장 작음 | 매우 큼(단열재 필수) | 중간 |
| 재사용 적합성 | 그을음(코킹) 세척 필요 | 금속 취성·누설 관리 | 그을음 거의 없음 |
| 자가가압 | 불가 — 헬륨 필요 | 가능 | 가능 — 헬륨 불필요 |
| 가격·조달 | 저렴, 공급망 성숙 | 비싸고 취급 인프라 고가 | 저렴(LNG 인프라 활용) |
| 대표 엔진 | Merlin, RD-180, 누리호 75tf | RS-25, Vulcain, LE-9, RL10 | Raptor, BE-4, KSLV-III 80tf |

> **읽는 법**: 어느 열도 전부 초록은 아니다. 수소는 Isp에서 이기고 밀도에서 지며, 케로신은 그 반대다. 메탄은 어디서도 1등이 아니지만 어디서도 크게 지지 않는다 — 이것이 재사용 시대에 메탄이 부상한 이유다.

### 2.4 밀도비추력 — 왜 1단과 상단의 답이 다른가

- $I_{sp}$만 보면 수소가 압승 → 그런데 왜 1단에 수소를 쓰는 발사체는 소수인가?

$$
\rho \cdot I_{sp} \quad (\text{밀도비추력}) = \text{"같은 부피의 탱크로 얼마나 큰 Δv를 살 수 있는가"}
$$

In [ ]:
# 06-density-isp-tradeoff.png 대신 인터랙티브 위젯으로 대체됨
propellant_isp_explorer()  # 위와 동일 위젯 재사용 — 밀도비추력까지 함께 표시됨

- **핵심 직관**: 추진제가 가벼우면 탱크가 커지고, 탱크가 커지면 구조 질량과 항력이 늘어남 → $I_{sp}$의 이득이 구조의 손해에 잡아먹힘
- **실제 설계는 이렇게 갈림**
  - **1단** — 중력손실·항력이 지배. 짧은 시간에 큰 추력 필요 → 밀도 높은 케로신·메탄·고체 유리
  - **상단** — 진공에서 천천히 오래 연소. 구조 손해 작음 → $I_{sp}$ 높은 수소 유리
  - **예외** — 델타 IV, 아리안 5·6, H3는 1단부터 수소, 단 고체 부스터로 이륙 추력 보충
  - → "수소 1단 + 고체 부스터"는 밀도 문제를 부스터로 메운 해법

> ※ ρ·Isp는 근사 계산값(혼합비 기준)으로 순위 비교용이며 절대 성능 지표가 아니다.

### 2.5 왜 지금 모두가 메탄으로 가는가

- 메탄 = 1960년대부터 알려진 연료 → 60년간 외면받다 2010년대 갑자기 주류화 (목표가 바뀌었기 때문)

- **그을음이 없다**: 케로신은 탄소 사슬이 길어 연소 시 그을음(코킹)이 냉각채널·터빈에 쌓임. 메탄은 CH₄(가장 단순한 탄화수소)라 침적물이 거의 없어 분해 세척 없이 재비행 가능
- **냉각 성능이 좋다**: 액체 메탄은 냉각 능력이 뛰어나 재생냉각에 유리 → 높은 연소실 압력을 견딜 수 있게 하고, 이는 더 높은 추력밀도로 이어짐
- **자가가압이 가능하다**: 엔진 열로 데운 메탄·산소 가스로 탱크 직접 가압 가능 → 헬륨 탱크·배관이 통째로 사라져 구조 단순화, 헬륨 조달 리스크 해소
- **온도가 산소와 비슷하다**: LOX(−183°C)와 LCH4(−162°C) 온도가 가까움 → 공통 격벽(common bulkhead) 탱크 설계가 쉬워지고 지상 설비도 단순화
- **싸고 흔하다**: LNG 공급망을 그대로 사용 → 조달가 낮음, 다빈도 발사에서 추진제 비용이 유의미해짐. 화성 현지자원(ISRU) 생산 가능성이라는 장기 논거도 존재

> **핵심**: 메탄의 장점 다섯 가지 중 넷은 '성능'이 아니라 '운용성'이다. 목표가 최대 성능이던 시절에는 메탄이 선택될 이유가 없었고, 목표가 반복 사용과 낮은 회당 비용으로 바뀌자 최적해가 바뀐 것이다. **기술 선택은 목적함수가 바뀌면 함께 바뀐다.**

### 2.6 하이퍼골릭 — 성능을 포기하고 확실성을 사는 선택

- 닿는 순간 스스로 불붙는 추진제. 점화 장치가 필요 없다는 단 하나의 장점이 나머지 모든 단점을 이긴 영역

- **무엇인가**
  - 대표 조합: NTO/MMH, UDMH — 산화제·연료가 접촉 즉시 자발 점화
  - 상온 저장 가능(극저온 아님), $I_{sp} \approx$ 320 s(케로신보다 낮음)
- **왜 쓰는가**
  - 점화기가 없어 고장 요소가 하나 사라짐
  - 수십 번의 재점화가 확실하게 보장
  - 수년간 충전 상태로 대기 가능 — 우주선 궤도조정·자세제어의 사실상 표준
- **무엇을 지불하는가**
  - 히드라진계는 강한 독성·발암성 → 취급 인력이 여압복 착용 필요
  - 지상 처리 비용·시간 크게 증가, 유럽 REACH 등 환경규제 압박 지속

| 적용 영역 | 사례 | 선택 이유 |
|---|---|---|
| 우주선 추진 | Apollo 기계선, 우주왕복선 OMS, 유인 캡슐 이탈·궤도조정 | 재점화 신뢰성이 생명과 직결 |
| 상단 엔진 | Proton 상단, 일부 중·인 발사체 상단 | 장기 코스트 단계 후 확실한 재점화 |
| 위성 추력기 | 정지궤도 위성 위치유지(전기추진으로 대체 진행 중) | 15년 이상 저장성 요구 |
| 군사 미사일 | 일부 액체 ICBM(러시아 계열) | 즉응성 — 다만 고체로 대체 추세 |

> **정리**: 추진제 선택은 성능표에서 최고치를 고르는 일이 아니라, 임무가 요구하는 제약조건(저장성·즉응성·재사용성·안전성) 중 무엇을 양보할 수 없는지를 먼저 정하는 일이다.

---

## PART III · 엔진 사이클

### 3.1 먼저 질문 — 추진제를 무슨 힘으로 밀어넣는가

- 연소실 압력이 높을수록 성능이 좋아짐 → 탱크에서 그보다 더 높은 압력으로 밀어넣어야 한다는 문제 발생

- **① 가압식(Pressure-fed)**: 탱크 자체를 고압 기체로 눌러 추진제를 밀어냄
![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s068_img001.png)

(출처 : AIAA / NASA LPTC 강의노트)

  - 장점: 회전 부품이 없어 단순·저렴·고신뢰. "적응 추력(thrust on demand)"
  - 한계: 탱크가 연소실보다 더 높은 압력을 견뎌야 함 → 벽 두꺼워짐, 연소실 압력 2 MPa 정도가 실용 상한
  - 용도: 착륙선, 궤도조정 추력기, 소형 상단
  - 사례 : Aerojet AJ10-190 (STS),  Aerojet AJ10-118 (Delta II), 대다수 RCS/ACS 시스템
| 상단 엔진 | 발사체 | 추력(ton) | Isp(sec) | Pcc(bar) | 연소시간(sec) | 추진제 |
|---|---|---|---|---|---|---|
| Aestus | Ariane 5 G, V | 3 | 324 | 11 | 1,100 | N2O4/MMH |
| AJ10-118K | Delta II | 4.45 | 321 | 8.96 | 444 | N2O4/A-50* |
| Kestrel | Falcon 1 | 3.4 | 317 | 9.3 | 418 | LOX/Kerosene |
| PSLV-4 | PSLV | 0.7 | 308 | 8.4 | 425 | N2O4/MMH |
| OME | Shuttle OMS | 2.73 | 316 | 8.62 | 1,250 | N2O4/MMH |
| Vega-Avum | Vega(개발중) | 0.25 | 315.5 | 20.4 | 667 | N2O4/MMH |

      - A-50 = 히드라진 50% + UDMH 50%

- **② 펌프식(Pump-fed)**: 터보펌프로 추진제를 가압, 탱크는 얇아도 됨
  - 장점: 연소실 압력 10~35 MPa 가능 → 추력밀도·$I_{sp}$ 대폭 향상, 탱크 경량화
  - 한계: 그 펌프를 무엇으로 돌릴 것인가라는 새로운 문제
  - 용도: 궤도발사체의 사실상 전부

> **터보펌프의 위력**: 대형 엔진의 터보펌프는 수만 마력을 낸다 — 자동차 엔진 수백 대분의 출력을 여행가방만 한 부피에서 낸다. 이 출력을 어디서 얻는가 — 답은 하나: 추진제의 일부를 미리 태워 그 가스로 터빈을 돌린다. 그 '미리 태운 가스를 어떻게 처리하는가'가 곧 엔진 사이클의 정의다.

![diagram](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/07-engine-cycles-overview.png)

| 사이클 | 터빈 구동 가스를 어떻게 하는가 | 대표 엔진 |
|---|---|---|
| 가스발생기(개방) | 따로 태우고 → 밖으로 버림 | Merlin, Vulcain, 누리호 75tf |
| 다단연소(폐쇄) | 따로 태우고 → 연소실로 돌려보냄 | RD-180, RS-25, YF-100 |
| 전량연소(FFSC) | 연료·산화제 전부를 예연소 → 전부 연소실로 | Raptor |
| 팽창기(폐쇄) | 태우지 않고 냉각으로 데운 연료로 터빈 구동 | RL10, Vinci, LE-9 |
| 전기펌프 | 배터리·모터로 직접 구동(예연소 없음) | Rutherford |

### 3.2 가스발생기 사이클 — 가장 널리 쓰인 실용해

- 터빈을 돌린 가스를 그냥 밖으로 버림 → 손해지만 대가로 설계가 극적으로 쉬워짐
- 추진제의 2\~5%가 추력에 기여하지 않고 버려짐 → $I_{sp}$ 손실 1~3%

- **왜 이렇게 널리 쓰이는가**
  - 연소기보다 낮은 압력이면 충분 → 펌프 요구 완화
  - 연료과잉으로 태워 터빈 입구 온도를 낮춤 → 터빈 재료 부담 작음
  - 예연소부와 주연소부가 분리되어 개별 시험·수정이 쉬움
  - 누리호 75 tf 엔진이 이 방식 — **후발국의 합리적 첫 선택**

| 구분 | 내용 |
|---|---|
| 이득 | 설계·시험 난이도가 낮고 개발 기간이 짧음. 부품 압력 요구가 낮아 제작비도 저렴 |
| 대가 | 버려지는 가스만큼 $I_{sp}$가 1~3% 낮음. 연소실 압력 상한도 낮아 추력밀도가 제한됨 |
| 판단 | 성능 손실이 '치명적이지 않은' 임무라면 여전히 최선. 발사체 1단처럼 $I_{sp}$ 민감도가 낮은 곳에 적합 |

- 단일추진제 가스발생기(Monopropellant GG)
  - 초기/원형 동력사이클
  - 성능은 수용 가능한 수준
  - 독립적인 단일추진제 제어로 신뢰성은 높지만, 제3추진제 탑재로 중량이 증가함
  - 사례
  - A-4 (V-2)
  - A-6 (Navaho-I)
  - A-7 (Redstone)
  - RD-107/108 (R-7 계열)
  - XLR99-RM-1 (X-15)
  - AR2-3A (F-104)

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s070_img002.png)


(출처 : AIAA / NASA LPTC 강의노트)
- 이원추진제, 단일 터보펌프(Bipropellant, Single TPA)
  - 단일추진제 GG 대비 성능 향상
  - 부트스트랩 시동
  - 제3추진제를 없애 추중비(T/W) 개선
  - 유사한 유체물성(밀도·점도)을 갖는 추진제 조합(LO2/RP-1)에 적합 — 공통 축 RPM 사용 가능
  - KSLV-II 엔진(75톤급, 7톤급)
  - F-1
  - Atlas MA-2, -3, -5, -5A
  - Navaho-II, -III
  - MC-1 / Fastrac / Merlin
  - S-3D → H-1 → RS-27

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s071_img002.png)

- 이원추진제 GG, 이중 터보펌프, 직렬터빈(Series Turbines)
  - 독립적인 추력·혼합비(MR) 제어 가능
  - 서로 다른 펌프 회전수를 요구하는 유체물성(LO2/LH2)의 추진제 조합에 적합
  - J-2 → J-2X

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s072_img002.png)

- 이원추진제 GG, 이중 터보펌프, 병렬터빈(Parallel Turbines)
  - 서로 다른 펌프 회전수를 요구하는 유체물성(LO2/LH2)의 추진제 조합에 적합
  - Vulcain
  - RS-68

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s073_img002.png)


### 3.3 다단연소 사이클 — 버리지 않고 되돌린다

- 터빈을 돌린 가스를 다시 연소기로 보냄 → 손실이 사라지지만 모든 부품이 훨씬 높은 압력을 견뎌야 함
- 모든 추진제가 주연소실을 통과해 추력에 기여 → 이론상 $I_{sp}$ 손실 0
- 대신 프리버너와 펌프는 연소실보다 더 높은 압력에서 작동해야 함

| 방식 | 설명 |
|---|---|
| 산화제과잉 | 산소가 남는 초고온 가스가 터빈을 지남. 금속이 산소 속에서 타는 문제를 잡아야 해 특수 합금·코팅 필수. 러시아가 독보적(RD-180, RD-191) |
| 연료과잉 | 터빈 가스가 상대적으로 덜 가혹함. 수소처럼 가벼운 연료여야 실용적. 미국이 채택(RS-25 우주왕복선 주엔진) |

| 구분 | 내용 |
|---|---|
| 이득 | $I_{sp}$ 손실 없음 + 높은 연소실 압력 → 같은 크기로 더 큰 추력, 더 높은 성능 |
| 대가 | 초고압·초고온 터빈, 산소 취급 재료 문제, 시스템 전체가 강하게 결합되어 한 곳을 바꾸면 전부 재설계 |
| 판단 | 선진 발사체의 표준이지만 개발 기간·비용·실패 위험이 가스발생기 대비 크게 증가 |


- 다단연소 사이클(Staged-combustion cycle)
  - 모든 추진제를 추력 생성에 활용
  - 고성능(추력, Isp, T/W)
  - 높은 Isp를 위해 고압 운용이 필요함
  - 신뢰성은 좋으나 고압 운용조건에서 세심한 관리가 필요
  - 고압 운용을 위해 통상 부스트펌프로 메인펌프 유입압력을 높여야 함
- 다단연소 사이클의 변형
  - 연료과잉형(FRSC)
  - LO2/LH2 추진제에 주로 사용. 예) Arroway 엔진(개발 중단)
  - 산화제과잉형(ORSC)
  - LO2/케로신 추진제에 주로 사용
  - NTO/UDMH도 사용
  - 완전유량형(FFSC)
  - LO2/LH2 추진제 기반 실험 시스템(IPD) 1건 개발됨
  - SpaceX의 Raptor가 대표적 사례
- 연료과잉 다단연소 사이클 — 이중 예연소기(Dual Preburners)
  - 독립적 혼합비·추력수준 스로틀링 가능
  - SSME

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s078_img002.png)

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s078_img004.png)

- 부스트펌프 미적용
- 부스트펌프 적용
- 연료과잉 다단연소 사이클 — 단일 예연소기, 이중 터보펌프
  - 일부 혼합비·추력수준 스로틀링 가능
  - 이중예연소기(DPFRSC) 대비 시스템이 단순해 신뢰성이 더 높음
  - RD-0120
  - LE-7
  - RS-30 ASE (DDT&E 미완료)
  - COBRA (DDT&E 미완료)

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s079_img003.png)

- 산화제과잉 다단연소 사이클 — 단일 터보펌프, 부스트펌프 적용
  - 신뢰성 우수
  - 산화제 과잉 환경에서 발화에 강한 소재 필요
  - 특수 코팅이 요구됨
  - 러시아에서만 독점적으로 사용됨
  - RD-253
  - RD-170 계열
  - RD-170, -171, -172
  - RD-180
  - RD-191
  - RS-84 (DDT&E 미완료)

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s080_img003.png)

- 완전유량 다단연소 사이클(Full-Flow SC Cycle) — 이중 터보펌프, 이중 예연소기
  - 시스템 복잡도 증가로 신뢰성 저하·비용 증가
  - 유동 관리가 복잡해 복잡한 과도상태·메인스테이지 제어가 필요
  - IPD (Integrated Powerhead Demonstrator)
  - Raptor (Starship)

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s081_img003.png)




### 3.4 전량연소 사이클(FFSC) — 이론적 정점

- 연료와 산화제 모두를 각자의 프리버너에서 태우고, 그 가스 전부를 연소기로 보냄
- 개념은 1960년대부터 있었지만 비행한 것은 Raptor가 처음
- 두 프리버너 가스가 모두 연소기로 — 액체 대 액체가 아니라 **기체 대 기체**로 만남

- **무엇이 좋아지는가**
  - 터빈이 둘로 나뉘어 각각의 부담이 줄고 온도가 낮아짐 → 수명 향상
  - 기체-기체 분사로 혼합이 빨라 연소가 안정적
  - 연소실 압력 30 MPa 이상 → 추력밀도 최고
  - 재사용의 관건인 수명·재점화 신뢰성이 함께 개선
- **왜 60년간 아무도 못 했나**
  - 펌프 2계통 + 프리버너 2계통 = 부품과 제어 루프가 배로 증가
  - 산화제과잉 초고온 가스 계통의 재료 문제를 여전히 풀어야 함
  - 시동 시퀀스가 극도로 복잡 — 두 계통의 균형이 어긋나면 폭발
- **무엇이 가능하게 했나**
  - 금속 적층제조(3D 프린팅)로 복잡한 냉각채널·부품 통합 실현
  - 고속 센서·연산으로 시동 과도구간을 실시간 제어
  - "만들고 터뜨리고 다시 만드는" 반복 시험을 감당하는 자금·조직

  ![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s082_img001.png)

- 완전유량 다단연소 사이클 — Raptor
- 출처 : https://www.reddit.com/r/spacex/comments/cxkrtb/detailed_diagram_of_the_raptor_engine_er26_gimbal/

### 3.5 예연소를 아예 하지 않는 두 가지 길

**팽창기 사이클(Expander)**

- 연소실 벽을 식히며 뜨거워진 연료(주로 수소)가 기체로 팽창 → 그 기체로 터빈을 돌린 뒤 연소기로 보냄. 태우는 과정 없음
- 장점: 예연소가 없어 매우 청정·안전, 부품 온도가 낮아 재점화 신뢰도 높음 — 상단 엔진의 오랜 표준(RL10은 1960년대부터 현역)
- 한계: 터빈 동력이 냉각 '면적'에서 나오는데 추력은 '부피'에 비례 → 엔진이 커질수록 면적(제곱)이 부피(세제곱)를 못 따라가 대형화가 막힘
- 우회로: 일본 LE-9는 데운 수소 일부를 터빈 후 버리는 '팽창기 블리드' 방식으로 대형화(150 tf) 달성 — 약간의 손실을 받아들여 크기 제약을 푼 사례

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s076_img002.png)

**전기펌프 사이클(Electric-pump)**

- 배터리로 전기모터를 돌려 펌프 구동 → 터빈도, 가스발생기도, 뜨거운 가스 배관도 전부 사라짐
- 장점: 구조가 압도적으로 단순 → 개발 기간 짧고 저렴. 추력을 전압으로 제어하므로 스로틀링이 정밀, 시동이 거의 즉각적
- 한계: 배터리가 무겁고, 연소가 진행돼도 그 질량이 줄지 않음 → 소형 발사체 규모를 넘어서면 불리
- 산업적 의미: Rocket Lab의 Rutherford는 터보펌프 기술 없이도 궤도발사가 가능함을 입증 — 소형발사체 진입장벽을 낮춘 결정적 혁신(12주차 연계)
![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s084_img002.png)

- http://www.popsci.com/rocket-labs-got-3d-printed-battery-powered-rocket-engine

![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s084_img004.png)


![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s084_img006.png)
(출처: H.-D. Kwak et al. 2018)

### 3.6 종합 비교 — 성능은 위험과 비용으로 산다

- 오른쪽으로 갈수록 성능이 높아지지만, 개발 기간·비용·실패 확률도 함께 상승

In [ ]:
# 08-cycle-performance-vs-risk.png 대신 인터랙티브 위젯으로 대체됨
engine_cycle_comparison()

| 사이클 | 예연소 가스 처리 | 연소실 압력 | 성능 | 개발 난이도 | 대표 엔진 |
|---|---|---|---|---|---|
| 가압식 | 없음 | 1~2 MPa | 낮음 | 매우 낮음 | 착륙선·추력기 |
| 전기펌프 | 없음(배터리) | ≈ 5 MPa | 중간 | 낮음 | Rutherford |
| 가스발생기 | 일부 버림 | 6~11 MPa | 중간(−1~3%) | 중간 | Merlin, 누리호 75tf |
| 팽창기 | 버리지 않음 | 4~6 MPa | 높음 | 중간 | RL10, Vinci |
| 다단연소 | 전량 회수 | 20~26 MPa | 매우 높음 | 높음 | RD-180, RS-25 |
| 전량연소(FFSC) | 전량 회수 ×2 | 30~35 MPa | 최고 | 매우 높음 | Raptor |

> **전략적 함의**: 후발 주자가 곧바로 오른쪽 끝(FFSC)으로 뛰면 일정과 예산이 무너진다. 대부분의 국가는 가스발생기로 진입해 신뢰성을 확보한 뒤 단계적으로 이동한다.

---


## PART IV · 운용 특성

### 4.1 재점화와 스로틀링 — 재사용이 엔진에 요구하는 것

- 일회용 발사체 엔진: 한 번 켜서 끝까지 태우면 됨
- 착륙하는 엔진: 전혀 다른 능력 필요

1. **이륙** — 전 엔진 최대 추력
2. **분리 후 정지** — 연소 중단, 잔압·냉각 관리
3. **역추진 점화** — 초음속·희박 대기에서 재점화
4. **착륙 연소** — 깊은 스로틀링으로 정밀 감속

- **재점화가 어려운 이유**
  - 점화원의 신뢰성 — 토치 점화기의 반복 작동 또는 자발점화 물질(TEA-TEB) 잔량이 재점화 횟수를 물리적으로 제한
  - 극저온 계통의 열충격 — 정지 중 데워진 배관에 다시 극저온 추진제가 흐름
  - 추진제 슬로싱과 침전 — 무추력 구간에서 액체가 탱크 바닥에 있다는 보장이 없어 얼리지(ullage) 처리 필요
- **스로틀링이 어려운 이유**
  - 유량을 줄이면 분사기 압력강하가 낮아져 연소가 불안정해짐 — 최소 추력에는 물리적 하한이 있음
  - 착륙 시점의 기체는 거의 비어 있어 매우 가벼움 — 엔진 하나의 최소 추력조차 기체 무게보다 클 수 있음
  - Falcon 9가 엔진 1기만으로 착륙하는 것도, 그 연소가 아슬아슬한 것도 이 때문

> **설계 사상의 전환**: 일회용 엔진의 목표함수는 '최대 성능'이었다. 재사용 엔진의 목표함수는 '충분한 성능 × 확실한 반복성'이다. 그래서 재사용 시대의 엔진은 종종 이전 세대보다 연소실 압력을 보수적으로 잡고, 대신 수명과 점검 주기를 늘린다. 성능 저하를 감수하고 운용성을 사는 것 — 5주차 재사용 경제성의 기술적 실체가 여기에 있다.

### 4.2 냉각과 수명 — 엔진의 진짜 한계는 열이다

- 연소실 벽은 3,000 K 넘는 가스에 노출 / 구리·합금의 녹는점은 1,300 K 안팎 → 이 모순을 푸는 것이 엔진 개발의 절반

| 방식 | 적용 | 원리 |
|---|---|---|
| 재생냉각 | 주력 방식 — 거의 모든 대형 엔진 | 연소실 벽 안에 수백 개의 미세 채널, 연소 전 추진제를 흘려 벽을 식힘. 흡수한 열은 버려지지 않고 연소실로 복귀 |
| 막냉각(Film) | 재생냉각의 보조 수단 | 벽면을 따라 연료를 얇게 흘려 보호막 형성. 국부적 연료과잉으로 성능 손실 감수 |
| 삭마냉각(Ablative) | 고체 노즐·소형 엔진·일회용 | 벽 재료가 스스로 타서 없어지며 열을 가져감. 단순·저렴하나 정의상 소모품 |

- **왜 재사용에서 냉각이 결정적인가**
  - 한 번 타는 엔진은 벽이 조금 손상돼도 임무가 끝남
  - 재사용 엔진은 매 비행마다 가열·냉각을 반복 — 열피로 균열이 수명을 정함
  - 재사용 엔진의 설계 지표는 '최대 추력'이 아니라 **'몇 사이클을 견디는가'**
  - 삭마냉각은 원리적으로 재사용과 양립 불가
- **제조기술이 성능을 푼다**
  - 과거: 구리 라이너에 수백 개 채널을 기계가공+전기도금 (고난도·고비용)
  - 현재: 금속 적층제조로 냉각채널을 내부에 품은 연소실을 한 덩어리로 출력
  - 부품 수 감소 → 접합부 감소 → 누설·고장 지점 감소 → 수명 증가
  - **제조혁신이 곧 성능혁신** — 신흥 기업의 경쟁력 원천

### 4.3 연소 불안정 — 엔진 개발이 오래 걸리는 진짜 이유

- 설계도상 완벽한 엔진이 시험대에서 수 초 만에 파괴됨 — 계산으로 예측되지 않는 현상이 남아 있기 때문

- **무슨 일이 일어나는가**
  - 연소실 안에서 압력이 주기적으로 진동 → 연료 공급의 리듬과 맞아떨어지면 서로를 증폭
  - 수 kHz의 고주파 진동에서는 진동하는 가스가 벽면 열전달을 폭발적으로 키워, 수백 밀리초 만에 연소실에 구멍이 뚫림 (폭발이 아니라 '녹아 내림'에 가까움)

> **고전 사례 — F-1 엔진(Saturn V)**: 1960년대 초, 연소 불안정으로 엔진이 반복적으로 파괴됨. 이론적 해석이 불가능해 결국 폭약을 터뜨려 인위적으로 교란을 준 뒤 얼마나 빨리 진정되는지를 재는 방식으로 접근. 분사기 배플(baffle) 형상을 수십 차례 바꿔가며 실험 — 사실상 시행착오. 이 문제 하나에 수년과 막대한 예산이 소요됨.

| 유형 | 주파수 | 원인 | 대응 |
|---|---|---|---|
| 저주파(Chugging) | 수십 Hz | 공급 계통과 연소실의 연성 진동 | 배관·오리피스 조정으로 비교적 해결 용이 |
| 중주파(Buzzing) | 수백 Hz | 분사기 공급 매니폴드의 음향 공진 | 매니폴드 설계 변경 |
| 고주파(Screeching) | kHz급 | 연소실 자체의 음향 모드와 연소의 결합 | 가장 위험 — 배플·음향공동으로 억제, 예측 곤란 |

> **정책적 함의**: 연소 불안정처럼 해석으로 예측되지 않는 문제가 남아 있는 한, 엔진 개발은 '충분한 시험 횟수'를 살 수 있는 예산과 설비를 가진 쪽이 이긴다. 이것이 시험설비 투자가 엔진 개발 예산의 큰 몫을 차지하는 이유다.

### 4.4 짐벌과 클러스터링 — 엔진을 몇 개 달 것인가

- 엔진 개수 = 단순한 추력 계산이 아니라 신뢰성·비용·개발전략이 얽힌 시스템 결정

- **추력벡터제어(TVC) — 방향은 어떻게 바꾸는가**
  - 짐벌: 엔진 전체를 유압·전동으로 기울임 (표준 방식)
  - 보조 수단: 가스발생기 배기 회전으로 롤 제어, 고체는 노즐 자체를 움직이거나 유체 분사
  - 4주차(항법유도제어)에서 제어기 설계와 함께 다시 다룸

- **소형 엔진 다수 vs 대형 엔진 소수**: 같은 500 tf 추력을 100 tf × 5기로 낼 것인가, 500 tf × 1기로 낼 것인가

| 비교 항목 | 소형 엔진 다수(클러스터링) | 대형 엔진 소수 |
|---|---|---|
| 개발 위험 | 낮음 — 작은 엔진을 먼저 완성 | 높음 — 대형 엔진 하나에 사업 전체가 걸림 |
| 엔진 고장 대응 | 일부 정지 후에도 임무 계속 가능(엔진아웃) | 고장 = 임무 상실 |
| 단가 | 같은 엔진을 많이 만들어 학습곡선 효과 극대화 | 생산 수량이 적어 단가 하락 여지 작음 |
| 시스템 복잡도 | 배관·제어 채널·화재 전파 위험 증가 | 단순 |
| 착륙 스로틀링 | 1기만 켜서 낮은 추력 구현 가능 | 최소 추력이 너무 커 착륙 곤란 |
| 대표 사례 | Falcon 9(9기), Super Heavy(33기), KSLV-III(9기) | Saturn V F-1(5기), Ariane 6 Vulcain(1기) |

> 재사용 시대에는 클러스터링이 뚜렷하게 우세하다. 착륙에 필요한 깊은 스로틀링, 엔진 양산에 따른 단가 하락, 엔진아웃 여유가 모두 같은 방향을 가리키기 때문이다.

### 4.5 엔진은 왜 발사체 개발의 임계경로인가

- 구조·전자·소프트웨어는 병행 개발 가능 / 엔진은 그렇지 않음 — 실물 시험 없이는 한 걸음도 못 나아감

![diagram](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/09-engine-development-timeline.png)

| 단계 | 기간 | 내용 |
|---|---|---|
| 개념·해석 | 1~2년 | 추진제·사이클·성능 목표 결정 |
| 연소기 시험 | 2~3년 | 분사기 형상 반복, 연소 불안정 제거 |
| 터보펌프 시험 | 2~3년 | 고속 회전체·베어링·씰, 폭발 사고 빈발 |
| 엔진 통합시험 | 2~3년 | 시동 시퀀스, 전체 계통 정합 |
| 인증·수명시험 | 1~2년 | 누적 연소시간·재점화 횟수 실증 |

- 단계 간 중첩을 감안해도 신규 대형 엔진 하나에 통상 **7~10년** — 발사체 전체 개발 기간을 사실상 이 일정이 결정

- **시험설비가 곧 국력**
  - 고공모사 시험설비는 진공 환경을 인위적으로 만들어야 해 건설비가 엔진 개발비에 육박
  - 터보펌프 시험대, 추진제 저장·처리, 소음·안전 이격거리까지 국가 인프라 문제
  - 설비가 없으면 해외 시험에 의존 — 일정과 기술주권 모두 종속
- **인력의 비가역성**
  - 연소 불안정 대응처럼 문서화되지 않는 암묵지가 핵심 역량
  - 사업이 중단되면 팀이 해체되고, 재개해도 처음부터 다시 시작
  - 미국이 F-1을 다시 만들지 못한 것, 유럽이 유인 발사 역량을 잃은 것이 같은 원리
- **정책 함의**
  - 엔진 개발은 단발 사업이 아니라 연속적 프로그램으로 설계해야 함
  - 10주차에서 본 각국의 '지속적 발주'는 산업 보호가 아니라 역량 유지 장치
  - 국가 발사 수요의 예측가능성 자체가 정책 수단

---

## PART V · 동향과 한국

### 5.1 세계 주력 엔진 지도

- 각 엔진의 사양은 그 나라가 무엇을 우선했는지를 그대로 보여줌

![diagram](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week03/images/10-world-engines-map.png)[width=50%]

| 엔진 | 국가 | 추진제 | 사이클 | 추력(지상) | Isp | 설계가 말하는 것 |
|---|---|---|---|---|---|---|
| Raptor 3 | 미국 | LOX/CH4 | 전량연소 | ≈280 tf | ≈350 s | 완전재사용·대량생산 지향 |
| Merlin 1D | 미국 | LOX/RP-1 | 가스발생기 | ≈85 tf | ≈282 s | 단순함으로 재사용 실증 |
| BE-4 | 미국 | LOX/CH4 | 다단연소(산화제과잉) | ≈250 tf | 비공개 | New Glenn·Vulcan 공용 |
| RS-25 | 미국 | LOX/LH2 | 다단연소(연료과잉) | ≈190 tf | ≈452 s | 성능 최우선, 고비용 |
| RL10 | 미국 | LOX/LH2 | 팽창기 | ≈11 tf | ≈465 s | 60년 넘게 현역인 상단 엔진 |
| Vulcain 2.1 | 유럽 | LOX/LH2 | 가스발생기 | ≈137 tf(진공) | ≈431 s | Ariane 6, 고체 부스터 병용 |
| LE-9 | 일본 | LOX/LH2 | 팽창기 블리드 | ≈150 tf(진공) | ≈425 s | 저비용·단순화 지향 |
| YF-100 | 중국 | LOX/RP-1 | 다단연소(산화제과잉) | ≈120 tf | ≈300 s | 창정 5·7호 계열 주력 |
| Rutherford | 뉴질랜드/미국 | LOX/RP-1 | 전기펌프 | ≈2.6 tf | ≈311 s(진공) | 소형발사체의 진입장벽 파괴 |
| 누리호 75tf급 | 한국 | LOX/RP-1 | 가스발생기 | ≈75 tf | ≈300 s급 | 자력 개발로 궤도투입 달성 |

> ※ 공개자료 기준 근사값으로, 버전·변형에 따라 달라진다. 진공 전용 엔진은 진공 기준임을 표기.

> **읽는 법**: 유럽·일본은 수소를 고수하며 성능을, 미국 신흥 기업은 메탄·케로신으로 운용성을, 중국은 다단연소로 성능과 자립을 동시에 추구한다. 사이클 열만 훑어도 각국의 산업 전략이 드러난다.



![](https://raw.githubusercontent.com/gabraxas/LVs-and-Policy/main/lecture/week02/assets/s085_img001.png)


### 5.2 엔진이 '작품'에서 '부품'으로 — 양산이라는 새 경쟁축

- 과거: 엔진은 연간 수 대를 정성껏 만드는 물건 / 지금: 생산율 자체가 경쟁력

- **무엇이 바뀌었나**
  - 재사용 발사체는 회수 후에도 엔진을 계속 만듦 — 함대를 키우고 예비품을 확보해야 하기 때문
  - SpaceX는 2026년 4월 기준 Raptor 계열을 600기 이상 생산했다고 발표
  - 적층제조·부품 통합으로 한 기당 제작 시간이 급감
  - 생산율이 곧 발사율의 상한이 됨
- **경제학적 의미**
  - 학습곡선이 작동하려면 누적 생산량이 있어야 함 — 연 2~3기 생산으로는 단가가 내려가지 않음
  - 클러스터링은 같은 엔진을 10배 더 만들게 하므로 학습곡선을 인위적으로 가속하는 설계 선택이기도 함
  - 12주차에서 다룰 발사가격 하락의 상당 부분이 여기서 나옴
- **후발국에 주는 함의**
  - 엔진을 '개발'하는 역량과 '싸게 반복 생산'하는 역량은 서로 다른 문제
  - 국내 발사 수요만으로는 학습곡선을 타기 어려운 누적 생산량 문제가 남음
  - 설계 단계에서 양산성을 고려했는가가 이후 단가를 좌우
  - 이 논점은 11주차 사업기획과 팀 보고서 원가 산정에서 직접 사용

### 5.3 한국의 선택 — 75톤 케로신에서 80톤 메탄으로

- 2025년 12월, 차세대발사체(KSLV-III)의 추진 방향이 변경됨 — 오늘 배운 내용의 종합 문제

| 항목 | 기존 계획(~2025) | 변경 후(2025.12 확정) | 오늘 강의로 읽으면 |
|---|---|---|---|
| 연료 | 케로신(RP-1) | 액체 메탄 | 재사용 시 그을음 제거 공정 회피 |
| 엔진 종류 | 1단·2단용 케로신 다단연소 2종 동시 개발 | 80 tf급 메탄 엔진 1종을 1·2단 공용 | 개발 대상을 절반으로 축소해 위험·비용 관리 |
| 사이클 | 다단연소사이클 목표 | 가스발생기 사이클 기반 접근 | 연료 전환과 사이클 전환을 동시에 시도하지 않음 |
| 기체 구성 | 1단 100 tf급 5기 | 1단 80 tf급 9기 · 2단 1기 | 클러스터링 확대로 착륙 스로틀링 여지 확보 |
| 목표 | 일회용 대형 발사체 | 1단 재사용 발사체 | 저비용·다빈도 발사 체계로 목표함수 전환 |

- **설계 논리로 읽기**
  - 목적함수가 재사용으로 바뀌자 메탄이 최적해가 됨
  - 메탄과 다단연소를 동시에 도전하지 않고 하나씩 — 기술위험 관리
  - 엔진 1종 공용화로 학습곡선과 시험설비 투자 효율을 함께 노림
- **남는 질문**
  - 성능(다단연소)을 미룬 대가는 언제, 어떤 형태로 청구되는가
  - 국내 발사 수요만으로 엔진 양산 학습곡선을 탈 수 있는가
  - 메탄 시험설비 신규 구축 비용은 어떤 기준으로 정당화되는가

> 출처: 우주항공청 보도자료(2025.12.22), 한국항공우주연구원 차세대발사체 사업 소개 — 세부 사양은 설계 진행에 따라 변경될 수 있다.

---

## PART VI · 심화 — 열역학으로 보는 추진성능의 근원

- 지금까지 배운 공식들 — $I_{sp} \propto \sqrt{T_c/M}$, $c^*$, $C_F$ — 은 결과만 제시됨 → 그 뒤의 열역학 제1법칙·기체분자운동론까지 보고 싶다면 아래 항목을 펼쳐볼 것
- **선택 사항 — 시험 범위 아님.** 유도 과정을 이해하면 "왜 수소가 유리한지", "왜 서머필드 기준이 그런 형태인지"를 암기가 아닌 증명으로 설명할 수 있게 됨
- 참고: Stanford AA284a *Advanced Rocket Propulsion* 강의노트 기반


<details>
<summary>💡 심화 1 — 왜 배기속도는 √(Tc/M)에 비례하는가 (1.4절 보충)</summary>

- **출발점 — 열역학 제1법칙(정상유동, 단열 노즐)**
  - 노즐 내부에서는 외부 열 유입도, 축일(shaft work)도 없음 → 전엔탈피(정체엔탈피)가 보존됨
  - $h_{t2} = h_{t1}$,  여기서 $h_t \equiv h + \frac{1}{2}u^2$ (엔탈피 + 운동에너지)
- **최대 배기속도 유도**
  - 연소실 정체엔탈피가 전부 운동에너지로 변환되는 극한(진공으로 무한 팽창, $h_e \to 0$)을 가정하면
  - $U_{e,max} = \sqrt{2h_{t2}} = \sqrt{2c_p T_{t2}} = \sqrt{\dfrac{2\gamma}{\gamma-1}\left(\dfrac{R_u}{M_w}\right)T_{t2}}$
  - 이 한 줄이 1.4절 "$I_{sp}$를 올리는 방법은 온도(T)와 분자량(M) 단 두 가지"라는 결론의 수학적 근거
- **왜 등엔트로피 가정이 필요한가**
  - 노즐 유동을 가역 단열 과정으로 볼 수 있어야 위 식이 성립 → 실제로는 마찰·열손실로 몇 % 손실 발생 (연소효율·노즐효율로 별도 관리)
  - 등엔트로피 관계식 $P/\rho^\gamma = const.$, $T_t/T = 1+\frac{\gamma-1}{2}M^2$ 등이 노즐 설계 전반의 계산 기반이 됨

</details>


<details>
<summary>💡 심화 2 — 비열비(γ)는 왜 이렇게 자주 등장하는가</summary>

- **기체분자운동론에서 압력을 유도하면**
  - 정육면체 용기 속 분자의 벽 충돌 운동량 변화를 합산 → $PV = n\left(\dfrac{M v_{rms}^2}{3}\right)$
  - 이상기체 상태방정식과 결합하면 분자의 평균 운동에너지 $K_{avg} = \frac{3}{2}k_B T$ → **온도란 결국 분자 운동에너지의 척도**
- **자유도(f)가 γ를 결정**
  - $c_v = \frac{f}{2}R$, $c_p = \left(\frac{f}{2}+1\right)R$ → $\gamma = \dfrac{c_p}{c_v} = \dfrac{f+2}{f}$
  - 단원자 기체(f=3) → γ≈1.67 / 연소가스 대부분을 이루는 이·다원자 기체(f 큼) → γ≈1.2~1.3
- **γ가 지배하는 4가지**
  - 음속: $a = \sqrt{\gamma R T}$ — 초크 유동·노즐 목 조건 전부 이 식에서 출발
  - 단열 압력-온도 관계 — 팽창 시 온도 강하폭 결정
  - $c^*$와 $C_F$ 공식 모두에 직접 등장 (심화 3, 4 참고)
  - 열역학 사이클(오토·브레이턴) 효율 — 터보펌프 예연소 사이클 설계와 연결
- **실무적 의미**: 연소가스의 γ는 추진제 조합·혼합비로 거의 정해짐 → 설계자가 직접 조절하는 변수가 아니라 추진제 선택의 '부산물'

</details>


<details>
<summary>💡 심화 3 — 특성속도 c*의 유도 (1.5절 보충)</summary>

- **초크(choked) 유동 질량유량식에서 출발**
  - 노즐 목(M=1)에서: $\dot m = \dfrac{A_* P_t}{\sqrt{T_t}}\sqrt{\dfrac{\gamma}{R}}\left(\dfrac{\gamma+1}{2}\right)^{-\frac{\gamma+1}{2(\gamma-1)}}$
  - 추력 정의 $F=\dot m V_e$와 목에 작용하는 힘 $F = P_t A_*$를 결합하면
  - $c^* \equiv V_e^* = \dfrac{P_t A_*}{\dot m} = \dfrac{\sqrt{\gamma}}{\gamma\sqrt{\left(\frac{2}{\gamma+1}\right)^{\frac{\gamma+1}{\gamma-1}}}}\sqrt{\dfrac{R_u}{M_w}T_t}$
- **왜 노즐 형상과 무관한가**
  - 식 전체가 연소실 조건($T_t$, $M_w$)과 γ로만 구성 — 팽창비·출구 조건은 전혀 등장하지 않음
  - γ 항은 γ 변화에 비교적 둔감 → **c*를 실질적으로 지배하는 것은 결국 $\sqrt{T_t/M_w}$** — 1.4절 배기속도 공식과 정확히 같은 물리
  - 그래서 c*는 "연소기 성적표"로 불림: 분사기 설계·혼합비·연소효율이 c*에 반영되고, 노즐이 얼마나 잘 뽑아내는지는 별개로 C_F가 담당

</details>


<details>
<summary>💡 심화 4 — 추력계수 C_F의 유도와 유동박리 서머필드 기준 (1.3, 1.5절 보충)</summary>

- **추력계수의 완전한 형태** (등엔트로피·완전기체 가정)
  - $C_F = \left\{\left(\dfrac{2\gamma^2}{\gamma-1}\right)\left(\dfrac{2}{\gamma+1}\right)^{\frac{\gamma+1}{\gamma-1}}\left[1-\left(\dfrac{P_e}{P_t}\right)^{\frac{\gamma-1}{\gamma}}\right]\right\}^{1/2} + \left(\dfrac{P_e}{P_t}-\dfrac{P_a}{P_t}\right)\dfrac{A_e}{A_t}$
  - 앞 항 = 팽창비가 만드는 "이상적" 추력계수 / 뒤 항 = 출구압-대기압 차이에 의한 보정 — 1.3절 과팽창·부족팽창 표가 바로 이 뒤 항의 부호 문제
- **유동박리 — 서머필드(Summerfield) 기준**
  - 과팽창이 심해지면 배기가 벽에서 떨어져 나가는 유동박리(flow separation) 발생 → 노즐 벽에 비대칭 측방향 하중, 구조 손상 위험
  - 경험적 박리 기준: $P_t/P_a > 16$ 조건에서 $P_e/P_a < 0.40$이면 박리 위험 (대형 노즐 최신 데이터 기준 0.286까지 낮아짐)
  - 실무 설계는 이 기준선 안쪽에서 팽창비를 정함 → 이론상 최적 팽창비보다 항상 보수적으로 설계되는 이유

</details>


<details>
<summary>💡 심화 5 — c*와 C_F를 곱하면 왜 Isp가 되는가</summary>

- **결합**
  - $T = \dot m \times \underbrace{c^*}_{\text{연소실 에너지}} \times \underbrace{C_F}_{\text{노즐 팽창 효율}}$
  - $I_{sp} \equiv \dfrac{T}{\dot m g_0} = \dfrac{c^* C_F}{g_0}$
  - 1.5절 "$I_{sp} g_0 = c^* \times C_F$"가 여기서 도출됨 — 성능을 연소기 몫(c*)과 노즐 몫(C_F)으로 완전히 분리할 수 있는 수학적 근거
- **관련 정의**
  - 임펄스밀도 $\delta \equiv T/\dot V_p = I_{sp}\rho_p$ — 밀도비추력(2.4절)이 여기서 나온 개념
  - 총 임펄스 $I_{tot} = \int_0^{t_b} T\,dt$, 평균추력 $\bar T = I_{tot}/t_b$
- **왜 이 분해가 실무적으로 중요한가**
  - 엔진 시험에서 예상보다 Isp가 낮으면, c*(연소실 성능)와 C_F(노즐 성능) 중 어느 쪽 문제인지 각각 측정해 원인을 즉시 좁힐 수 있음 → 1.5절 "성적표를 두 과목으로 나누는" 실무 관행의 이론적 뿌리

</details>


## 요약 및 정리

1. **$I_{sp}$는 온도와 분자량의 함수** — 배기속도는 $\sqrt{T_c/M}$에 비례. 수소가 최고 $I_{sp}$인 것은 뜨거워서가 아니라 가벼워서. 노즐은 그 에너지를 방향 있는 속도로 바꾸는 장치이며, 팽창비 하나가 1단과 상단 노즐 형상의 차이를 모두 설명
2. **추진제 선택은 절충의 문제** — 전 항목에서 우수한 조합은 없음. 1단은 밀도($\rho \cdot I_{sp}$), 상단은 $I_{sp}$가 지배. 메탄의 부상은 성능이 아니라 운용성이 목적함수가 되었기 때문
3. **엔진 사이클은 예연소 가스를 어떻게 처리하는가로 갈림** — 버리면 가스발생기, 되돌리면 다단연소, 전부 되돌리면 전량연소, 태우지 않으면 팽창기·전기펌프. 성능이 올라갈수록 개발 난이도와 비용이 함께 상승
4. **재사용은 엔진의 목표함수를 바꿈** — 최대 성능이 아니라 재점화·깊은 스로틀링·긴 수명이 설계를 지배. 냉각 설계와 제조기술이 그 한계를 정함
5. **엔진은 발사체 개발의 임계경로이자 정책의 대상** — 연소 불안정처럼 예측 불가능한 문제가 남아 있어 시험 횟수를 살 수 있는 예산과 설비가 승부를 가름. 엔진 개발은 단발 사업이 아니라 연속 프로그램으로 설계되어야 함

---

## 자가진단 퀴즈 (Self-Assessment)

<details>
<summary>Q1. 수소(LOX/LH2)의 진공 비추력이 케로신(LOX/RP-1)보다 높은 이유는 연소온도가 더 높기 때문인가?</summary>

아니다. LOX/LH2의 연소온도(≈3,000K)는 오히려 LOX/RP-1(≈3,600K)보다 낮다. Isp가 높은 이유는 배기가스의 평균 분자량이 훨씬 낮기 때문이다($V_e \propto \sqrt{T_c/M}$에서 M의 영향이 지배적).
</details>

<details>
<summary>Q2. 왜 대부분의 발사체는 1단과 상단에 서로 다른 노즐(또는 다른 엔진)을 쓰는가?</summary>

노즐은 한 고도에서만 최적 팽창(Pe=Pa)이 된다. 1단은 지상~저고도에서 작동하므로 팽창비가 작은(ε≈10~20) 노즐이 필요하고, 상단은 진공에서 작동하므로 팽창비가 큰(ε≈80~300) 노즐이 유리하다. 하나의 노즐로 전 고도를 최적화할 수 없기 때문이다.
</details>

<details>
<summary>Q3. c*와 C_F를 나누어 평가하는 실무적 이유는?</summary>

엔진 성능이 목표에 못 미쳤을 때 원인을 특정하기 위해서다. c*(연소기 성적)가 낮으면 분사기·연소 문제이고, C_F(노즐 성적)가 낮으면 노즐 문제다. 이 분리 덕분에 연소기 시험과 노즐 시험을 별도로 수행해 문제를 국소화할 수 있다.
</details>

<details>
<summary>Q4. 다단연소 사이클이 가스발생기 사이클보다 이론상 Isp 손실이 없는 이유는?</summary>

가스발생기는 터빈을 돌린 가스를 밖으로 버리지만(추진제의 2~5% 손실), 다단연소는 그 가스를 다시 주연소실로 돌려보내 모든 추진제가 최종적으로 추력에 기여하기 때문이다. 대신 프리버너와 펌프가 훨씬 높은 압력에서 작동해야 하는 대가를 치른다.
</details>

<details>
<summary>Q5. KSLV-III가 케로신 대신 메탄을, 다단연소 대신 가스발생기를 선택한 것은 어떤 논리인가?</summary>

목표함수가 '일회용 대형 발사체'에서 '1단 재사용 발사체'로 바뀌었기 때문이다. 메탄은 재사용 시 그을음 제거 공정을 피할 수 있어 재사용에 유리하고, 가스발생기는 다단연소보다 개발 위험이 낮다. 연료 전환(케로신→메탄)과 사이클 전환(다단연소 시도)을 동시에 하지 않고 하나씩 진행해 기술위험을 관리한 선택이다.
</details>

## 정책 토론 — 기술 선택은 곧 정책 선택이다

다음 주 정책메모 및 팀 사업기획보고서 작성 시 아래 논점을 근거로 활용할 수 있다.

**Q1. 사이클 선택의 정치경제**
고성능 사이클은 개발 기간과 실패 확률을 함께 키운다. 임기가 정해진 정부와 예산 주기가 있는 국가 사업에서, 이 위험을 감수한 선택은 어떤 조건에서 가능한가? 반대로 위험 회피가 기술 정체로 이어지는 것은 어떻게 막는가?

**Q2. 메탄 전환의 기회비용**
메탄 전환은 재사용에 유리하지만 기존 케로신 시험설비·공급망·인력 숙련의 상당 부분을 새로 쌓아야 한다. 이미 축적된 자산을 버리는 결정을 정당화하는 기준은 무엇이어야 하는가?

**Q3. 시험설비의 공공성**
엔진 시험설비는 건설비가 크고 가동률이 낮다. 이를 국가가 지어 민간에 개방하는 방식과, 민간이 자체 구축하도록 지원하는 방식 중 어느 쪽이 산업 생태계에 유리한가? (9주차 미국 사례와 비교)

**Q4. 양산 규모의 딜레마**
학습곡선 효과는 누적 생산량에서 나오는데, 국내 발사 수요는 제한적이다. 수출·해외 발사서비스 없이 엔진 단가를 낮출 방법이 있는가? 없다면 그 사실은 발사체 사업의 타당성 평가에 어떻게 반영되어야 하는가?

---

## 참고문헌 및 더 읽을거리

★ 표시는 이번 주 필수 확인 자료.

**교재**
- ★ Sutton, G. P. & Biblarz, O., *Rocket Propulsion Elements*, 9th ed., Wiley — 3~5장(노즐·연소), 6장(액체추진제), 10~11장(터보펌프·엔진 사이클)
- Huzel, D. K. & Huang, D. H., *Modern Engineering for Design of Liquid-Propellant Rocket Engines*, AIAA Progress Series Vol. 147
- Humble, R. W. et al., *Space Propulsion Analysis and Design*, McGraw-Hill

**강의자료**
- ★ Karabeyoglu, A., *AA284a Advanced Rocket Propulsion*, Stanford University — 노즐 이론, 액체·고체 추진, 궤적 강의 시리즈
- TU Delft AE4S01, *Thermal Rocket Propulsion* — 이상 노즐 이론부터 화학추진 시스템 설계까지
- NASA Glenn Research Center, *Beginner's Guide to Rocketry* — 추력식·노즐 개념의 입문 설명

**산업·정책 자료**
- ★ 우주항공청, 「차세대발사체, 재사용발사체로 개발 확정」 보도자료, 2025. 12. 22.
- 한국항공우주연구원, 차세대발사체 사업 소개 (www.kari.re.kr) — 기체 구성 및 목표 성능
- 한국추진공학회지 — 메탄 추진제 및 재사용 엔진 관련 국내 연구 동향
- SpaceX, Raptor 엔진 공개 사양 및 생산 현황 발표 자료 (2024~2026)

**서술 구조 참고**
- Leishman, J. G., ["Rockets & Launch Vehicle Performance,"](https://eaglepubs.erau.edu/introductiontoaerospaceflightvehicles/chapter/rocket-performance/) *Introduction to Aerospace Flight Vehicles*, Embry-Riddle Aeronautical University (open textbook)

> **수치에 관한 주의**: 본 강의노트의 성능 수치($I_{sp}$, 추력, 연소실 압력 등)는 공개자료 기준 근사값이며, 실제 엔진의 공식 제원과는 차이가 있을 수 있다. 정량 과제 수행 시에는 각 기관의 공식 공개자료를 직접 확인할 것.

---

**다음 주(4주차) 예고**: 이 엔진을 실제 발사체에 얹는 문제 — 구조·유도제어·상승궤적·발사장 안전을 다룬다.